When a person states their symptoms, stating whether a symptom is present or not, the system will recommend a series of extra questions, evidences that in case of being true or false will contribute to the system diagnosing a person. Such recommendation of questions will be based on the features with no registered answer (set to zero) that would either affirm the diagnosis or reject it and point toward another diagnosis.

The recommendation system is based on the algrebraic difference between on each index comparing both patient vector and diagnosis profile vector

In [ ]:
import pandas as pd
import numpy as np
import json
from google import genai

In [ ]:
client = genai.Client(api_key = "Super secret key")

In [ ]:
all_features = pd.read_json("/content/drive/MyDrive/Disease_engine_project/release_evidences.json")

In [ ]:
all_features = all_features.T

In [ ]:
all_features = all_features.drop(columns = ["code_question", "question_fr"])

In [ ]:
all_features = all_features.reset_index(drop=True)

In [ ]:
all_features = all_features.drop(columns = ["is_antecedent"])

The following section formats the questions in a way that can be better understood by the LLM by matching a question with its answer

In [ ]:
binary = all_features[all_features["data_type"] == "B"]

In [ ]:
multiple_choice = all_features[all_features["data_type"] == "M"]

In [ ]:
single_choice = all_features[all_features["data_type"] == "C"]

In [ ]:
type(single_choice.iloc[0]["value_meaning"])

dict

In [ ]:
row_list=[]
for i in range(0, single_choice.shape[0]):
  row = single_choice.iloc[i]
  main_name = row["name"]
  main_question = row["question_en"]
  new_row = {}
  for j in row["possible-values"]:
    new_row = {}
    row_name = main_name
    row_question = main_question
    row_name = row_name + "_" + str(j)
    if row["default_value"] == 0:
      row_question = row_question + " - Answer: " + str(j)
    else:
      row_question = row_question + " - Answer: " + str(row["value_meaning"][j]["en"])
    new_row["code"] = row_name
    new_row["question"] = row_question
    new_row["value"] = 0
    new_row["type"] = 'C'
    row_list.append(new_row)

In [ ]:
for i in range(0, multiple_choice.shape[0]):
  row = multiple_choice.iloc[i]
  main_name = row["name"]
  main_question = row["question_en"]
  new_row = {}
  for j in row["possible-values"]:
    new_row = {}
    row_name = main_name
    row_question = main_question
    row_name = row_name + "_" + str(j)
    if row["default_value"] == 0:
      row_question = row_question + " - Answer: " + str(j)
    else:
      row_question = row_question + " - Answer: " + str(row["value_meaning"][j]["en"])
    new_row["code"] = row_name
    new_row["question"] = row_question
    new_row["value"] = 0
    new_row["type"] = 'M'
    row_list.append(new_row)

In [ ]:
binary

,name,question_en,default_value,value_meaning,possible-values,data_type
0,E_91,Do you have a fever (either felt or measured w...,0,{},[],B
2,E_53,"Do you have pain somewhere, related to your re...",0,{},[],B
8,E_159,Did you lose consciousness?,0,{},[],B
10,E_129,"Do you have any lesions, redness or problems o...",0,{},[],B
17,E_154,Is your skin much paler than usual?,0,{},[],B
...,...,...,...,...,...,...
218,E_195,Do you live in the suburbs?,0,{},[],B
219,E_183,Do you live in a rural area?,0,{},[],B
220,E_224,Do you have family members who have had lung c...,0,{},[],B
221,E_223,Are there members of your family who have been...,0,{},[],B


In [ ]:
for i in range(0, binary.shape[0]):
  row = binary.iloc[i]
  main_name = row["name"]
  main_question = row["question_en"]
  new_row = {}
  new_row["code"] = main_name
  new_row["question"] = main_question
  new_row["value"] = 0
  new_row["type"] = 'B'
  row_list.append(new_row)

In [ ]:
pd.DataFrame(row_list).head()

,code,question,value
0,E_59_0,How fast did the pain appear? - Answer: 0,0
1,E_59_1,How fast did the pain appear? - Answer: 1,0
2,E_59_2,How fast did the pain appear? - Answer: 2,0
3,E_59_3,How fast did the pain appear? - Answer: 3,0
4,E_59_4,How fast did the pain appear? - Answer: 4,0


In [ ]:
prompt_data = str(row_list)

The following cell sets up "rules" for the LLM to get the desired result

In [ ]:
prompt1 = "The following data has this format: [{\"code\": code, \"question\": question and answer to the question, \"value\": value, \"type\": type}]: "
prompt2 = "Return this list in the same valid JSON format following the set of rules I will state:"
rule_prompt_B = "\nIf a question has its value field set to \'B\': Change the value of the \'value\' field to 1 if the information provides enough evidence for the answer of that question to be yes, change it to -1 if the information provides enough information for the answer to that question to be no, leave it as 0 if there isn't enough evidence for answering such question."
rule_prompt_C = "\nIf a question has its value field set to \'C\': Change the value of the \'value\' field to 1 if the information provides enough evidence for affirming the answer of the given question, change it to -1 if the information provides enough evidence for refuting the answer of the given question, leave it as 0 if there isn't enough evidence for affirming or refuting. Amongst all the questions that share the same code section E_*, and whose type is \'C\', only one option per code (E_*_V_* or E_*_*) can be 1 at the time, that is, setting to 1 an option (E_*_V_* or E_*_*) automatically refutes all the other options that share the same E_* code."
rule_prompt_M = "\nIf a question has its value field set to \'M\': Change the value of the \'value\' field to 1 if the information provides enough evidence for affirming the answer of the given question, change it to -1 if the information provides enough evidence for refuting the answer of the given question, leave it as 0 if there isn't enough evidence for affirming or refuting. Amongst all the questions that share the same code section E_*, and whose type is \'M\', more than one option per code E_*_V_* can be 1 at the same time, that is, setting to 1 an option E_*_V_* does NOT automatically refute all the other options that share the same E_* code, therefore, non explicitly mentioned answers should not be set to -1 unless the user explicitly discards them."
rule_closer = "nOnce done, double check your work to ensure that the product matches the expected format in every single row.\nPrompt: "


In [ ]:
user_prompt = "The pain took its sweet time to appear, and is kinda intense. You know, sure it is my poor diet and antecedents of lung cancer. Nevertheless, that doesn't explain the sharp pain in both of my biceps. But hey, at least I haven't had asthma and both my knees feel excelent."

In [ ]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt1 + prompt_data + "\n" + prompt2 + rule_prompt_B + rule_prompt_C + rule_prompt_M + rule_closer + user_prompt
)
print(response.text)

```json
[
  {"code": "E_59_0", "question": "How fast did the pain appear? - Answer: 0", "value": -1, "type": "C"},
  {"code": "E_59_1", "question": "How fast did the pain appear? - Answer: 1", "value": -1, "type": "C"},
  {"code": "E_59_2", "question": "How fast did the pain appear? - Answer: 2", "value": -1, "type": "C"},
  {"code": "E_59_3", "question": "How fast did the pain appear? - Answer: 3", "value": -1, "type": "C"},
  {"code": "E_59_4", "question": "How fast did the pain appear? - Answer: 4", "value": -1, "type": "C"},
  {"code": "E_59_5", "question": "How fast did the pain appear? - Answer: 5", "value": -1, "type": "C"},
  {"code": "E_59_6", "question": "How fast did the pain appear? - Answer: 6", "value": -1, "type": "C"},
  {"code": "E_59_7", "question": "How fast did the pain appear? - Answer: 7", "value": -1, "type": "C"},
  {"code": "E_59_8", "question": "How fast did the pain appear? - Answer: 8", "value": -1, "type": "C"},
  {"code": "E_59_9", "question": "How fast di

In [ ]:
data = json.loads(response.text[8:-3].replace("\'", "\"").replace(", M\"", ", \"type\": \"M\""))

In [ ]:
LLM_data = pd.DataFrame(data)

In [ ]:
LLM_data

,code,question,value,type
0,E_59_0,How fast did the pain appear? - Answer: 0,-1,C
1,E_59_1,How fast did the pain appear? - Answer: 1,-1,C
2,E_59_2,How fast did the pain appear? - Answer: 2,-1,C
3,E_59_3,How fast did the pain appear? - Answer: 3,-1,C
4,E_59_4,How fast did the pain appear? - Answer: 4,-1,C
...,...,...,...,...
967,E_195,Do you live in the suburbs?,0,B
968,E_183,Do you live in a rural area?,0,B
969,E_224,Do you have family members who have had lung c...,1,B
970,E_223,Are there members of your family who have been...,0,B


In [ ]:
codes = LLM_data["code"].to_list()

In [ ]:
values = LLM_data["value"].to_list()

In [ ]:
vector = pd.DataFrame(columns = codes)

In [ ]:
vector = vector.T

In [ ]:
vector[0] = values

In [ ]:
vector = vector.T

Vector resulting from the given prompt

In [ ]:
vector

,E_59_0,E_59_1,E_59_2,E_59_3,E_59_4,E_59_5,E_59_6,E_59_7,E_59_8,E_59_9,...,E_200,E_199,E_121,E_120,E_142,E_195,E_183,E_224,E_223,E_5
0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
from cosine_sim_engine import Cosine_sim_classifier

In [ ]:
classifier = Cosine_sim_classifier()

In [ ]:
valid_vector = classifier.extract_features(vector)

/usr/local/lib/python3.12/dist-packages/cosine_sim_engine/classify.py:123: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_record = new_record.fillna(0)


Using the built classifier for predicting the top 5 possible diagnosis

In [ ]:
classifier.similarity(valid_vector)

,Diagnosis,Scores
0,Possible NSTEMI / STEMI,0.000590
1,Pulmonary embolism,0.000539
2,Cluster headache,0.000539
3,Epiglottitis,0.000538
4,Spontaneous rib fracture,0.000537
